# 🏥 Frontend — Modelo Predictivo de IAAS (v5)
## Random Forest Balanceado | Criterios IAAS validados | Umbral = 0.5

**Tres modos de uso:**
1. 🔍 **Predicción individual** — ingresa un caso directamente en el código
2. 📊 **Batch sin etiquetas** — predice desde un Excel sin columna IAAS
3. 📈 **Evaluación batch** — analiza con etiquetas reales (matriz de confusión + gráficos)

**Archivos necesarios:**
- `modelo_iaas.pkl` — generado por `modelo_random_forest_IAAS_exportar_v5.ipynb`
- `set_prueba_con_predicciones.xlsx` (solo para el modo 3)

---
## 📦 PASO 1: Instalación y librerías

In [ ]:
!pip install -q openpyxl xlsxwriter scikit-learn

import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
from google.colab import files
warnings.filterwarnings('ignore')
print('✅ Librerías listas')

---
## ⚙️ PASO 2: Funciones de preprocesamiento (coherentes con v5)

In [ ]:
CRITERIOS_VALIDOS = [
    # A - Bacteriemia/Fungemia asociada a CVC
    'A.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'A.I.a.e2.Hipotermia igual o menor a 36 °C axilar',
    'A.I.a.e3.Hipotensión',
    'A.I.a.e4. Taquicardia o bradicardia',
    'A.I.a.e5.Apnea en pacientes menores de un año',
    'A.I.a.e6.Eritema y exudado en sitio de inserción del CVC',
    'A.I.b.e1.Detección en uno o más set de hemocultivos periféricos de un microorganismo patógeno no relacionado con otra infección activa en otra localización por el mismo agente',
    'A.I.b.e2.Detección de microorganismo comensal en al menos dos sets de hemocultivos periféricos tomados en sitios anatómicos diferentes no relacionado con otra infección activa en otra localización por el mismo agente',
    'A.I.b.e3.Detección de microorganismo comensal en al menos un set de hemocultivos periféricos y en cultivo de punta de catéter retirado por sospecha clínica de infección, no relacionado con otra infección activa en otra localización por el mismo agente',
    # B - ITU asociada a sonda vesical
    'B.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'B.I.a.e2.Síntomas irritativos vesicales (tenesmo vesical, urgencia miccional, polaquiuria, disuria, dolor suprapúbico)',
    'B.I.a.e3.Dolor costo vertebral a la palpación o espontáneo',
    'B.I.a.e4.Alteración nueva del estado de conciencia en pacientes de 65 o más años',
    'B.I.b.e1.Leucocituria de acuerdo con los valores de referencia del laboratorio que procesó la muestra tomada',
    'B.I.b.e2.Presencia de placas de pus',
    'B.I.b.e3.Presencia de piocitos',
    'B.I.c.e1.Cultivo de orina con no más de dos microorganismos, en el que al menos uno de ellos tiene recuento de más de 100.000 UFC/ml',
    # C - ISQ
    'C.I.a.e1.Presencia de pus (exudado purulento) en el sitio de incisión quirúrgica, incluido el sitio de la salida de drenaje por contrabertura, con o sin cultivos positivos',
    'C.II.a.e1.Fiebre igual o mayor a 38 °C no atribuible a otra causa',
    'C.II.a.e2.Sensibilidad o dolor en la zona de la incisión quirúrgica',
    'C.II.a.e3.Aumento de volumen localizado en la zona de la incisión quirúrgica',
    'C.II.a.e4.Eritema o calor local en la zona de la incisión quirúrgica',
    'C.II.a.e5.La incisión es deliberadamente abierta por un integrante del equipo de salud1 con presencia de exudado que, sin tener aspecto de pus, se describe como turbio, serohemático o seropurulento',
    'C.II.a.e6.Aislamiento de microorganismo en cultivo obtenido con técnica aséptica de la incisión o tejido subcutáneo',
    # D - Diarrea infecciosa
    'D.I.a.e1.Paciente tiene dos o más deposiciones líquidas dentro de 12 horas con o sin otra sintomatología, no atribuible a causas no infecciosas',
    'D.I.b.e1.Si se cuenta con agente etiológico identificado, no hay evidencias que el microorganismo se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    'D.II.a.e1.Paciente presenta un episodio de deposiciones líquidas o disgregadas',
    'D.II.b.e1.Crecimiento de microorganismo patógeno entérico en cultivo de deposiciones o en muestra de hisopado rectal',
    'D.II.b.e2.Microorganismo patógeno entérico detectado por cualquier medio que no sea cultivo',
    'D.II.c.e1.No hay evidencias que el microorganismo se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    # E - C. difficile
    'E.I.a.e1.Presencia de más de una deposición líquida en 12 horas',
    'E.I.a.e2.Presencia de 3 o más deposiciones disgregadas o líquidas en 24 horas',
    'E.II.a.e1.Muestra de deposición positiva a toxina de C. difficile por cualquier técnica de laboratorio, o aislamiento de cepa productora de toxina detectada en deposición por cultivo u otro medio incluida biología molecular',
    # F - Neumonía
    'F.I.a1.e1.Infiltrado',
    'F.I.a1.e2.Condensación',
    'F.I.a1.e3.Cavitación',
    'F.I.a2.e1.Infiltrado nuevo o progresión de uno existente',
    'F.I.a2.e2.Condensación',
    'F.I.a2.e3.Cavitación',
    'F.I.b.e1.Fiebre mayor o igual a 38 °C axilar',
    'F.I.b.e2.Leucopenia (<4.000 leucocitos/mm3) o leucocitosis (>12.000 leucocitos/mm3)',
    'F.I.b.e3.Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.I.b.e4.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC/ml o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    'F.II.a.e1.Infiltrado nuevo o progresión de uno existente',
    'F.II.a.e2.Condensación',
    'F.II.a.e3.Cavitación',
    'F.II.a.e4.Neumatoceles',
    'F.II.b.e1.Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.II.c.e1.Temperatura corporal inestable',
    'F.II.c.e2.Leucopenia (11.000 leucocitos/mm3) con desviación a izquierda (Mayor o igual a 10% de baciliformes o formas más inmaduras)',
    'F.II.c.e3.Aparición de expectoración purulenta, o cambios en las características, o aumento de la cantidad, o aumento en los requerimientos de aspiración de secreciones',
    'F.II.c.e4.Sibilancias, estertores o roncus',
    'F.II.c.e5.Inestabilidad hemodinámica',
    'F.II.c.e6.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC/ml1010 o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    'F.III.a.e1.Presenta Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.III.b.e1.Aparición de expectoración, aumento o cambio en las características, o aumento de los requerimientos de aspiración o succión de secreciones',
    'F.III.b.e2.Hemoptisis',
    'F.III.b.e3.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC3/ml o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    # G - IRA viral
    'G.I.a.e1.Fiebre igual o mayor a 38 °C axilar o hipotermia sin otra causa reconocible',
    'G.I.a.e2.Leucopenia (<4.000 leucocitos/mm3) o leucocitosis (>11.000 leucocitos/mm3)',
    'G.I.a.e3.Tos',
    'G.I.a.e4.Aparición o incremento de producción de expectoración',
    'G.I.a.e5.Roncus',
    'G.I.a.e6.Sibilancias',
    'G.I.a.e7.Distrés respiratorio o síndrome de dificultad respiratoria',
    'G.I.a.e8.Apnea',
    'G.I.a.e9.Bradicardia',
    'G.I.a.e10.Imagen pulmonar no presente al ingreso compatible con infección viral',
    'G.I.b.e1.Detección de agente viral respiratorio por cualquier técnica de laboratorio',
    'G.I.c.e1.No hay evidencias que el agente viral respiratorio se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    # H - COVID-19
    'H.I.a.e1.Fiebre igual o mayor a 37,8 °C axilar',
    'H.I.a.e2.Perdida brusca y completa del olfato (anosmia)',
    'H.I.a.e3.Perdida brusca y completa del gusto (ageusia)',
    'H.I.a.e4.Tos o estornudos',
    'H.I.a.e5.Congestión nasal',
    'H.I.a.e6.Disnea o dificultad respiratoria',
    'H.I.a.e7.Taquipnea',
    'H.I.a.e8.Odinofagia',
    'H.I.a.e9.Mialgia',
    'H.I.a.e10.Debilidad general o fatiga',
    'H.I.a.e11.Dolor torácico',
    'H.I.a.e12.Calofríos',
    'H.I.a.e13.Diarrea',
    'H.I.a.e14.Anorexia o nauseas o vómitos',
    'H.I.a.e15.Cefalea',
    'H.I.b.e1.Prueba PCR para SARS-CoV-2 positiva',
    'H.I.b.e2.Prueba de antígenos para SARS-CoV-2 positiva',
    'H.I.c.e1.Tomografía de tórax con opacidades bilaterales múltiples en vidrio esmerilado, con distribución pulmonar periférica y baja sin otra causa conocida',
    'H.II.a.e1.Fiebre igual o mayor a 37,8 °C axilar',
    'H.II.a.e2.Perdida brusca y completa del olfato (anosmia)',
    'H.II.a.e3. Perdida brusca y completa del gusto (ageusia)',
    'H.II.a.e4.Tos o estornudos',
    'H.II.a.e5.Congestión nasal',
    'H.II.a.e6.Disnea o dificultad respiratoria',
    'H.II.a.e7.Taquipnea',
    'H.II.a.e8.Odinofagia',
    'H.II.a.e9.Mialgia',
    'H.II.a.e10.Debilidad general o fatiga',
    'H.II.a.e11.Dolor torácico',
    'H.II.a.e12.Calofríos',
    'H.II.a.e13.Diarrea',
    'H.II.a.e14.Anorexia o nauseas o vómitos',
    'H.II.a.e15.Cefalea',
    'H.II.b.e1.Prueba PCR para SARS-CoV-2 positiva',
    'H.II.b.e2.Prueba de antígenos para SARS-CoV-2 positiva',
    # I - Endometritis
    'I.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'I.I.a.e2.Sensibilidad uterina o subinvolución uterina',
    'I.I.a.e3.Loquios de aspecto purulento o cambio en la evolución de su aspecto o aumento de mal olor',
    'I.II.a.e1.La paciente tiene un cultivo de fluido o tejido endometrial positivo obtenidos intraoperatoriamente, por punción uterina o por aspirado uterino con técnica aséptica hasta 10 días posterior al parto',
    # J - Meningitis/Ventriculitis
    'J.I.a.e1.Detección de microorganismos (cultivo, test molecular) en líquido cefalorraquídeo (LCR) recolectado con técnica aséptica para fines diagnósticos o terapéuticos',
    'J.II.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'J.II.a.e2.Dolor de cabeza',
    'J.II.a.e3.Signos meníngeos',
    'J.II.a.e4.Signos de nervios craneales',
    'J.II.a.e5.Modificación cualitativa o cuantitativa de conciencia.',
    'J.II.a.e6.Apnea (en menores de un año)',
    'J.II.a.e7.Bradicardia (en menores de un año)',
    'J.II.b.e1.LCR con aumento de glóbulos blancos o en los niveles de proteínas o con descenso de nivel de glucosa según rangos reportados por laboratorio local',
    'J.II.b.e2.Microorganismo observados en tinción de Gram del LCR',
    'J.II.b.e3.Identificación en uno o más set de hemocultivos periféricos de un microorganismo no relacionado con otra infección activa en otra localización por el mismo agente',
    # K - Endoftalmitis
    'K.I.a.e1.Paciente presenta identificación de un microorganismo en muestra tomada con técnica aséptica de cámara anterior, posterior o humor vítreo',
    'K.II.a.e1.Dolor ocular',
    'K.II.a.e2.Visión borrosa',
    'K.II.a.e3.Hipopion',
    'K.II.b.e1.Como consecuencia de los signos y síntomas, el médico inicia terapia antibiótica de 2 o más días de duración',
]
CRITERIOS_SET = {c.strip().lower() for c in CRITERIOS_VALIDOS}
CATEGORIA_CRITERIO = {
    'a': 'bacteriemia_cvc', 'b': 'itu_sonda', 'c': 'isq',
    'd': 'diarrea_infecciosa', 'e': 'c_difficile', 'f': 'neumonia',
    'g': 'ira_viral', 'h': 'covid19', 'i': 'endometritis',
    'j': 'meningitis', 'k': 'endoftalmitis',
}
COLS_CRITERIO = ['Criterio 1', 'Criterio 2', 'Criterio 3', 'Criterio 4', 'Criterio 5']
VALORES_NO_CRITERIO = {'sin hallazgos iaas', 'no aplica', 'nan', 'sin_dato', ''}

def es_criterio_valido(valor):
    v = str(valor).strip().lower()
    return v not in VALORES_NO_CRITERIO and v in CRITERIOS_SET

def calcular_features_criterios_fila(fila):
    conteo = 0
    categorias = {}
    for col in COLS_CRITERIO:
        val = fila.get(col, '')
        if es_criterio_valido(val):
            conteo += 1
            pref = str(val).strip()[0].lower() if str(val).strip() else ''
            if pref in CATEGORIA_CRITERIO:
                cat = CATEGORIA_CRITERIO[pref]
                categorias[cat] = categorias.get(cat, 0) + 1
    n_cat = len(categorias)
    cat_principal = max(categorias, key=categorias.get) if categorias else 'ninguna'
    return conteo, n_cat, cat_principal

def limpiar_texto(val, col=None):
    v = str(val).strip().lower()
    if v in ['nan', '']: v = 'sin_dato'
    return v

def categorizar_dispositivo(tipo):
    tipo = str(tipo).lower()
    if 'catéter venoso central' in tipo or 'cvc' in tipo: return 'cvc'
    elif 'sonda vesical' in tipo: return 'sonda_vesical'
    elif 'ventilación' in tipo: return 'ventilacion_mecanica'
    elif 'epicutáneo' in tipo: return 'cateter_epicutaneo'
    elif 'hemodiálisis' in tipo or 'hemodialisis' in tipo: return 'cateter_hemodialisis'
    elif 'umbilical' in tipo: return 'cateter_umbilical'
    elif 'derivativa' in tipo or 'dve' in tipo or 'dvp' in tipo: return 'derivativa_ventricular'
    else: return 'otro'

def tiene_araisp_fn(x):
    return 0 if pd.isna(x) or str(x).strip().lower() in ['nan','','no','no aplica'] else 1

def preparar_fila(caso, paquete):
    """Convierte un dict de caso en vector de features para el modelo."""
    enc  = paquete['encoders']
    feats_cat  = paquete['features_categoricas']
    feats_num  = paquete['features_numericas']
    feats_all  = paquete['features']
    cats_valid = paquete['categorias_validas']

    # Normalizar textos
    for col in ['Servicio','Procedencia','Destino','Tipo de invasivo']:
        caso[col] = limpiar_texto(caso.get(col, 'sin_dato'))
    _mapa_tipo = {'epicutáneo':'catéter epicutáneo','dve':'derivativa ventricular externa',
         'dvp':'derivativa ventricular peritoneal'}
    caso['Tipo de invasivo'] = _mapa_tipo.get(caso['Tipo de invasivo'], caso['Tipo de invasivo'])

    # Features criterios
    n_crit, n_cat, cat_p = calcular_features_criterios_fila(caso)
    caso['n_criterios']         = n_crit
    caso['n_cat_criterios']     = n_cat
    caso['categoria_principal'] = cat_p
    caso['tiene_araisp']        = tiene_araisp_fn(caso.get('ARAISP',''))
    caso['tipo_dispositivo_cat']= categorizar_dispositivo(caso.get('Tipo de invasivo',''))

    # Año
    fi = caso.get('Fecha de ingreso', None)
    try:
        caso['anio'] = pd.to_datetime(fi, errors='raise').year if fi is not None and not pd.isna(fi) else 2025
    except Exception:
        caso['anio'] = 2025

    row = {}
    for col in feats_cat:
        val = caso.get(col, 'sin_dato')
        val_str = str(val)
        if val_str not in [str(c) for c in enc[col].classes_]:
            val_str = str(enc[col].classes_[0])  # fallback
        row[col + '_enc'] = enc[col].transform([val_str])[0]
    for col in feats_num:
        row[col] = caso.get(col, 0)

    return pd.DataFrame([row])[feats_all]

def nivel_riesgo(prob):
    if prob < 0.25:  return '🟢 BAJO',   '#2ecc71'
    if prob < 0.50:  return '🟡 MODERADO','#f39c12'
    if prob < 0.75:  return '🔴 ALTO',    '#e74c3c'
    return '🚨 MUY ALTO', '#c0392b'

print('✅ Funciones de preprocesamiento listas')

---
## 📂 PASO 3: Cargar modelo (`modelo_iaas.pkl`)

In [ ]:
# ── Carga modelo_iaas.pkl directamente desde Google Drive ──
import gdown, os

FILE_ID_PKL = '1OO3IOYFwtA-Xm3FZ03SYGlWAoS4BtiIU'
nombre_pkl  = 'modelo_iaas.pkl'

if not os.path.exists(nombre_pkl):
    url = f'https://drive.google.com/uc?id={FILE_ID_PKL}'
    gdown.download(url, nombre_pkl, quiet=False)
else:
    print(f'✅ Usando archivo ya descargado: {nombre_pkl}')

with open(nombre_pkl, 'rb') as f:
    paquete = pickle.load(f)

modelo   = paquete['modelo']
UMBRAL   = paquete.get('umbral_optimo', 0.5)
metricas = paquete.get('metricas', {})

print(f'✅ Modelo cargado: {nombre_pkl}')
print(f'   Umbral óptimo  : {UMBRAL:.4f}')
print(f'   Features       : {paquete["features"]}')
print(f'   AUC-ROC        : {metricas.get("auc_roc","N/A")}')
print(f'   Accuracy       : {metricas.get("accuracy","N/A")}')
print(f'   Sensibilidad   : {metricas.get("recall","N/A")}')

# Archivo batch (se reutiliza en pasos 5, 6 y 7)
ARCHIVO_BATCH = 'set_prueba_con_predicciones.xlsx'

---
## 🔍 PASO 4: Predicción individual
Modifica los valores del diccionario `caso` y ejecuta la celda.

In [ ]:
# ════════ INGRESA LOS DATOS DEL CASO ════════
caso = {
    # ── Datos demográficos / administrativos
    'Servicio'          : 'UCI 2',
    'Procedencia'       : 'Emergencia',
    'Destino'           : 'UCI 2',
    'Fecha de ingreso'  : '2025-03-15',
    'ARAISP'            : None,  # None = no tiene ARAISP

    # ── Dispositivo invasivo
    'Tipo de invasivo'  : 'Catéter venoso central',

    # ── Criterios IAAS (pega el texto exacto del listado oficial,
    #    o pon 'Sin hallazgos IAAS' si no aplica)
    'Criterio 1': 'F.I.b.e1.Fiebre mayor o igual a 38 °C axilar',
    'Criterio 2': 'F.I.b.e4.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC/ml o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    'Criterio 3': 'Sin hallazgos IAAS',
    'Criterio 4': 'Sin hallazgos IAAS',
    'Criterio 5': 'Sin hallazgos IAAS',
}

# ════════ PREDICCIÓN ════════
X_caso = preparar_fila(caso.copy(), paquete)
prob   = modelo.predict_proba(X_caso)[0][1]
pred   = 'IAAS' if prob >= UMBRAL else 'NO IAAS'
nivel, color = nivel_riesgo(prob)

# ── Detalle de criterios detectados
crit_detectados = []
for c in COLS_CRITERIO:
    val = caso.get(c, '')
    if es_criterio_valido(val):
        crit_detectados.append(f'  ✓ {val[:70]}')

print('\n' + '='*55)
print(f'  RESULTADO: {pred}')
print(f'  Probabilidad de IAAS : {prob*100:.1f}%')
print(f'  Nivel de riesgo      : {nivel}')
print(f'  Umbral aplicado      : {UMBRAL:.2f}')
print('='*55)
n_crit, n_cat, cat_p = calcular_features_criterios_fila(caso)
print(f'  Criterios válidos    : {n_crit}/5')
print(f'  Categorías IAAS      : {n_cat}  ({cat_p})')
if crit_detectados:
    print('  Criterios activos:')
    for c in crit_detectados: print(c)
else:
    print('  Sin criterios IAAS válidos detectados')

# ── Gráfico barra de probabilidad
fig, ax = plt.subplots(figsize=(8, 1.6))
cmap = LinearSegmentedColormap.from_list('riesgo', ['#2ecc71','#f1c40f','#e74c3c'], N=100)
ax.barh(0, prob, height=0.5, color=cmap(prob), edgecolor='#333')
ax.barh(0, 1-prob, left=prob, height=0.5, color='#ecf0f1', edgecolor='#ccc')
ax.axvline(UMBRAL, color='#2c3e50', lw=2, ls='--', label=f'Umbral {UMBRAL:.2f}')
ax.set_xlim(0, 1); ax.set_yticks([])
ax.set_xlabel('Probabilidad de IAAS', fontsize=10)
ax.set_title(f'{pred}  |  {prob*100:.1f}%  |  {nivel}', fontsize=11, color=color, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

---
## 📤 PASO 5: Cargar archivo Excel para predicción en batch
Sube el archivo con los casos a evaluar. Puede o no tener la columna `IAAS`.

In [ ]:
# ── Carga set_prueba_con_predicciones.xlsx directamente desde Google Drive ──
import gdown, os

FILE_ID_XLSX = '1wR6OKLboJxZkKeyy8p946Zyd0gwlNAvN'
ARCHIVO_BATCH = 'set_prueba_con_predicciones.xlsx'

if not os.path.exists(ARCHIVO_BATCH):
    url = f'https://drive.google.com/uc?id={FILE_ID_XLSX}&export=download'
    gdown.download(url, ARCHIVO_BATCH, quiet=False, fuzzy=True)
else:
    print(f'✅ Usando archivo ya descargado: {ARCHIVO_BATCH}')

df_batch = pd.read_excel(ARCHIVO_BATCH)
TIENE_IAAS_REAL = 'IAAS' in df_batch.columns

print(f'✅ Archivo cargado: {ARCHIVO_BATCH}')
print(f'   Filas          : {len(df_batch)}')
print(f'   Columnas       : {list(df_batch.columns)}')
print(f'   Tiene col IAAS : {TIENE_IAAS_REAL}')
if TIENE_IAAS_REAL:
    print(f'   Distribución IAAS: {df_batch["IAAS"].value_counts().to_dict()}')

---
## 📊 PASO 6: Predicción en batch

In [ ]:
# Columnas a excluir (predicciones anteriores)
COLS_EXCLUIR = ['Prob_IAAS (%)', 'Prediccion (umbral 0.55)', 'Prediccion (umbral 0.68)',
                'Prediccion', 'Acierto']
df_trabajo = df_batch.drop(columns=[c for c in COLS_EXCLUIR if c in df_batch.columns], errors='ignore')

resultados   = []
y_prob_lote  = []
y_pred_lote  = []

for _, row in df_trabajo.iterrows():
    caso_dict = row.to_dict()
    X_row = preparar_fila(caso_dict.copy(), paquete)
    prob  = modelo.predict_proba(X_row)[0][1]
    pred  = 'IAAS' if prob >= UMBRAL else 'NO IAAS'
    y_prob_lote.append(prob)
    y_pred_lote.append(pred)
    resultados.append(prob * 100)

df_trabajo = df_trabajo.copy()
df_trabajo['Prob_IAAS (%)']            = [round(p,1) for p in resultados]
df_trabajo[f'Prediccion (umbral {UMBRAL:.2f})'] = y_pred_lote

# Mostrar resumen
from collections import Counter
cnt = Counter(y_pred_lote)
print(f'✅ Predicciones completadas: {len(df_trabajo)} casos')
print(f'   IAAS    : {cnt["IAAS"]}')
print(f'   NO IAAS : {cnt["NO IAAS"]}')

# Gráficos de distribución
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist([p*100 for p in y_prob_lote], bins=20, color='#3498db', edgecolor='white')
ax1.axvline(UMBRAL*100, color='#e74c3c', lw=2, ls='--', label=f'Umbral {UMBRAL*100:.0f}%')
ax1.set_xlabel('Probabilidad IAAS (%)'); ax1.set_ylabel('Frecuencia')
ax1.set_title('Distribución de probabilidades'); ax1.legend()
colores = ['#e74c3c' if p == 'IAAS' else '#2ecc71' for p in y_pred_lote]
etiquetas = list(cnt.keys()); conteos = [cnt[e] for e in etiquetas]
cols_pie = ['#e74c3c' if e == 'IAAS' else '#2ecc71' for e in etiquetas]
ax2.pie(conteos, labels=etiquetas, colors=cols_pie, autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribución de predicciones')
plt.suptitle(f'Resultados batch — {len(df_trabajo)} casos | Umbral {UMBRAL:.2f}', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 💾 PASO 7: Descargar predicciones en Excel

In [ ]:
nombre_salida = 'predicciones_iaas_v5.xlsx'
df_trabajo.to_excel(nombre_salida, index=False)
files.download(nombre_salida)
print(f'✅ Descargado: {nombre_salida}')

---
## 📈 PASO 8: Evaluación con etiquetas reales (requiere columna `IAAS`)
> ⚠️ Solo ejecutar si el archivo cargado en el Paso 5 tiene la columna `IAAS`.

In [ ]:
if not TIENE_IAAS_REAL:
    print('⚠️  El archivo no tiene columna IAAS. Sáltate este paso.')
else:
    def limpiar_iaas(val):
        v = str(val).strip().upper()
        if v in ['SI','SÍ','1','IAAS','TRUE','YES','S']: return 1
        return 0

    y_true = df_batch['IAAS'].apply(limpiar_iaas).values
    y_prob_arr = np.array(y_prob_lote)
    y_pred_bin = (y_prob_arr >= UMBRAL).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_bin).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    espec= tn / (tn + fp) if (tn + fp) > 0 else 0
    auc  = roc_auc_score(y_true, y_prob_arr)

    print('\n📋 MÉTRICAS DE EVALUACIÓN')
    print('='*40)
    print(f'  Sensibilidad  : {sens:.4f}')
    print(f'  Especificidad : {espec:.4f}')
    print(f'  AUC-ROC       : {auc:.4f}')
    print(f'  VN={tn}  FP={fp}  FN={fn}  VP={tp}')
    print('='*40)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Matriz de confusión
    ax = axes[0]
    cm = np.array([[tn, fp], [fn, tp]])
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    etiq = ['No IAAS (0)', 'IAAS (1)']
    ax.set_xticks([0,1]); ax.set_xticklabels(etiq, rotation=20)
    ax.set_yticks([0,1]); ax.set_yticklabels(etiq)
    total = cm.sum()
    labels_cm = [['VN','FP'],['FN','VP']]
    for i in range(2):
        for j in range(2):
            c = cm[i, j]
            ax.text(j, i, f'{labels_cm[i][j]}\n{c}\n({c/total*100:.1f}%)',
                    ha='center', va='center', fontsize=11,
                    color='white' if c > cm.max()/2 else 'black')
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusión | Umbral {UMBRAL:.2f}\n'
                 f'Sens={sens:.2f}  Espec={espec:.2f}', fontweight='bold')

    # ── Curva ROC
    ax2 = axes[1]
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_prob_arr)
    ax2.plot(fpr_arr, tpr_arr, color='#3498db', lw=2, label=f'AUC = {auc:.4f}')
    ax2.plot([0,1],[0,1],'--', color='gray', lw=1)
    ax2.scatter([1-espec],[sens], color='#e74c3c', s=100, zorder=5,
                label=f'Umbral {UMBRAL:.2f} (FPR={1-espec:.2f}, TPR={sens:.2f})')
    ax2.set_xlabel('1 - Especificidad (FPR)')
    ax2.set_ylabel('Sensibilidad (TPR)')
    ax2.set_title('Curva ROC', fontweight='bold')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.suptitle('Evaluación del Modelo en Set de Prueba', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    # ── Distribución de probabilidades por clase real
    fig2, ax3 = plt.subplots(figsize=(9, 4))
    probs_pos = [y_prob_arr[i]*100 for i in range(len(y_true)) if y_true[i]==1]
    probs_neg = [y_prob_arr[i]*100 for i in range(len(y_true)) if y_true[i]==0]
    ax3.hist(probs_neg, bins=20, alpha=0.65, color='#2ecc71', label='No IAAS (real)', edgecolor='white')
    ax3.hist(probs_pos, bins=20, alpha=0.65, color='#e74c3c', label='IAAS (real)', edgecolor='white')
    ax3.axvline(UMBRAL*100, color='#2c3e50', lw=2, ls='--', label=f'Umbral {UMBRAL*100:.0f}%')
    ax3.set_xlabel('Probabilidad predicha (%)'); ax3.set_ylabel('Frecuencia')
    ax3.set_title('Distribución de probabilidades por clase real', fontweight='bold')
    ax3.legend(); plt.tight_layout(); plt.show()

    print(f'\n{classification_report(y_true, y_pred_bin, target_names=["No IAAS","IAAS"])}')

---
## 🌳 PASO 9: Importancia de variables y métricas del modelo

In [ ]:
NOMBRES_LEGIBLES = {
    'Servicio_enc'             : 'Servicio clínico',
    'Procedencia_enc'          : 'Procedencia',
    'Destino_enc'              : 'Destino',
    'tipo_dispositivo_cat_enc' : 'Tipo dispositivo',
    'categoria_principal_enc'  : 'Categoría IAAS principal',
    'n_criterios'              : 'N° criterios IAAS válidos',
    'n_cat_criterios'          : 'N° categorías IAAS distintas',
    'tiene_araisp'             : 'Tiene ARAISP',
    'anio'                     : 'Año de ingreso',
}

feats  = paquete['features']
import_vals = modelo.feature_importances_
df_imp = pd.DataFrame({'Feature': feats, 'Importancia': import_vals})
df_imp['Nombre'] = df_imp['Feature'].map(lambda x: NOMBRES_LEGIBLES.get(x, x))
df_imp = df_imp.sort_values('Importancia')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(df_imp['Nombre'], df_imp['Importancia'],
               color=plt.cm.RdYlGn(df_imp['Importancia'] / df_imp['Importancia'].max()))
for bar, val in zip(bars, df_imp['Importancia']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importancia (Gini)'); ax.set_xlim(0, df_imp['Importancia'].max() * 1.2)
ax.set_title('Importancia de Variables — Modelo IAAS v5', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print('\n📋 Métricas del modelo entrenado (set de prueba 20%):')
for k, v in metricas.items():
    if k not in ['n_train','n_test','n_total','tn','fp','fn','tp']:
        print(f'   {k:<14}: {v}')
if all(k in metricas for k in ['tn','fp','fn','tp']):
    print(f'   VN={metricas["tn"]}  FP={metricas["fp"]}  FN={metricas["fn"]}  VP={metricas["tp"]}')